In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Inside Airbnb data processing").getOrCreate()

In [0]:
listings = spark.read.csv("/Volumes/bootcamp_students/zachy_w_brett_coleman/airbnbdata/listings.csv.gz",
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE"
    )

In [0]:
review_locations = listings.select(listings.review_scores_location)
review_locations.show()

In [0]:
listings \
    .select(listings.review_scores_location) \
    .show()

In [0]:
# Find listings with a rating higher than 4.5 using filter
high_score_listings = listings \
    .filter(listings.review_scores_location > 4.5) \
    .select('id', 'price', 'name', 'review_scores_location')

high_score_listings.show(20, truncate=False)

In [0]:
# Drop NULL values
high_score_listings.dropna().show(20, truncate=False)

In [0]:
high_score_listings.schema['price']

In [0]:
# Price is a string so we need to transform it to a float
from pyspark.sql.functions import regexp_replace, col

price_num_df = listings \
    .withColumn('price_num', regexp_replace('price', '[$,]', '').cast('float')) \

price_num_df.schema['price_num']

In [0]:
# The data looks much nicer now
price_num_df \
 .select('price_num', 'name') \
    .show(20, truncate=False)

In [0]:
# Find some hidden gems
price_num_df.filter((price_num_df.price_num < 100) & (price_num_df.review_scores_location > 4.5)) \
    .select('name', 'price', 'review_scores_location') \
    .show(truncate=False)

In [0]:
# Find all distinct values of a column
listings \
    .select(listings.property_type, listings.room_type) \
        .distinct() \
            .show(truncate=False)

##Exercises

In [0]:
listings.printSchema()

In [0]:

# 1. Get a non-null picutre URL for any property ("picutre_url" field)
# Select any non-null picture URL

In [0]:
picture_url_df = listings.select(listings.picture_url).dropna()
picture_url_df.show(1, truncate=False)

In [0]:
# 2. Get number of properties that get more than 10 reiews per month


In [0]:
ten_k_reviews_df = listings.filter(listings.reviews_per_month > 10).count()
ten_k_reviews_df

In [0]:

# 3. Get a property that has more bathrooms than bedrooms

In [0]:
more_bathrooms_df = listings.filter(listings.bathrooms > listings.bedrooms)
display(more_bathrooms_df.limit(1))

In [0]:

# 4. Get 10 properties where the price is greater than 5000. Collect the result as a python list

In [0]:
# Price is a string so we need to transform it to a float
from pyspark.sql.functions import regexp_replace, col

price_num_df = listings \
    .withColumn('price_num', regexp_replace('price', '[$,]', '').cast('float'))

res = price_num_df.filter((col('price_num') > 50000)) \
    .select('name', 'price') \
    .collect()

res

In [0]:
# 6. Get a list of properties with the following characteristics:
# * price < 150 or more than one bathroom
# Use the "|" operator to implement the OR operator

In [0]:
from pyspark.sql.functions import regexp_replace

listings.filter(
    (regexp_replace('price', '[$,]', '').cast('float') < 150) &
    (listings.number_of_reviews > 20) &
    (listings.review_scores_rating > 4.5)
) \
.select('name', 'price', 'number_of_reviews', 'review_scores_rating') \
.show(truncate=False)

In [0]:
# 7. Get the highest listing price in this dataset
# Consider using the "max" function

In [0]:
from pyspark.sql.functions import max
price_highest_df = listings \
    .withColumn('price_num', regexp_replace('price', '[$,]', '').cast('float'))
price_highest_df\
    .select(max('price_num'))\
    .show()

In [0]:
# 8. Get the name and a price of the property with the highest number of reviews per month
# Try to use "collect" method to get the price first, and then use it in a "filter" call

In [0]:
# 9. Get the number or hosts in the dataset

In [0]:
listings\
    .select('host_name')\
    .distinct()\
    .count()

In [0]:
# 10. Get listings with first reviews in 2024
# Consider using the "year" function from "pyspark.sql.functions"

In [0]:
from pyspark.sql.functions import year

listings.filter(
    year(listings.first_review) == 2024
)\
.select('name', 'first_review')\
.show(10, truncate=False)